# LIGHTGBM + FILTERED FEATURES THEO MEAN SHAP VÀ THEO MEDIAN SHAP

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold

In [ ]:
app_train_fe = pd.read_parquet("../Data/app_train_fe.parquet")
app_test_fe = pd.read_parquet("../Data/app_test_fe.parquet")
mean_shap = pd.read_csv("../Data/mean_shap.csv")

In [ ]:
# Tách features và target
X_train = app_train_fe.drop(columns=["TARGET"])
X_test = app_test_fe.copy()
y_train = app_train_fe["TARGET"]

In [ ]:
# Loại bỏ SK_ID_CURR vì chỉ là ID định danh, không có giá trị dự báo
# Giữ lại để dùng sau (submit Kaggle, OOF predictions)
features = [col for col in X_train.columns if col not in ["SK_ID_CURR", "TARGET"]]
print(f"Số features đưa vào model: {len(features)}")

# Lấy danh sách cột category để khai báo với LightGBM
# LightGBM xử lý category trực tiếp mà không cần One-Hot hay Label Encoding
cat_cols = X_train[features].select_dtypes(include="category").columns.tolist()
print(f"Số cột category: {len(cat_cols)}")

In [ ]:
# ==============================================================================
# LỌC FEATURES THEO NGƯỠNG MEAN SHAP (GIỮ LẠI 66 FEATURES)
# ==============================================================================

# Lấy ngưỡng = mean của mean_abs_shap
shap_threshold_mean = mean_shap["mean_abs_shap"].mean()
print(f"Ngưỡng SHAP (Mean): {shap_threshold_mean:.6f}")

# Lấy danh sách features có mean |SHAP| >= ngưỡng mean
selected_features_mean = mean_shap[mean_shap["mean_abs_shap"] >= shap_threshold_mean][
    "feature"
].tolist()

print(f"Số features ban đầu: {len(features)}")
print(f"Số features sau lọc (Mean): {len(selected_features_mean)}")
print(f"Số features bị loại (Mean): {len(features) - len(selected_features_mean)}")

# Cập nhật cat_cols cho bộ features lọc theo Mean
cat_cols_filtered_mean = [col for col in cat_cols if col in selected_features_mean]
print(f"Số cột category sau lọc (Mean): {len(cat_cols_filtered_mean)}")
print("-" * 50)


# ==============================================================================
# BỔ SUNG: LỌC FEATURES THEO NGƯỠNG MEDIAN SHAP (TRUNG VỊ)
# ==============================================================================

# Lấy ngưỡng = median của mean_abs_shap
# Thích hợp khi dữ liệu SHAP có một vài feature quá vượt trội (outliers) làm lệch giá trị Mean
shap_threshold_median = mean_shap["mean_abs_shap"].median()
print(f"Ngưỡng SHAP (Median): {shap_threshold_median:.6f}")

# Lấy danh sách features có mean |SHAP| >= ngưỡng median
selected_features_median = mean_shap[
    mean_shap["mean_abs_shap"] >= shap_threshold_median
]["feature"].tolist()

print(f"Số features sau lọc (Median): {len(selected_features_median)}")
print(f"Số features bị loại (Median): {len(features) - len(selected_features_median)}")
# Cập nhật cat_cols cho bộ features mới theo Median
cat_cols_filtered_median = [col for col in cat_cols if col in selected_features_median]
print(f"\nSố cột category sau lọc (Median): {len(cat_cols_filtered_median)}")

In [ ]:
# Giữ nguyên siêu tham số mặc định của bạn
params_default = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "n_estimators": 1500,
    "random_state": 42,
    "n_jobs": -1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "verbose": -1,
}

# Cấu hình 2 tập dữ liệu đầu vào dựa trên kết quả lọc trước đó
strategies = {
    "Mean SHAP": {
        "features": selected_features_mean,
        "cat_cols": cat_cols_filtered_mean,
    },
    "Median SHAP": {
        "features": selected_features_median,
        "cat_cols": cat_cols_filtered_median,
    },
}

# Dictionary dùng để lưu trữ kết quả đầu ra của từng chiến lược
evaluation_results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Vòng lặp tự động chạy qua từng chiến lược lọc
for name, config in strategies.items():
    current_features = config["features"]
    current_cat_cols = config["cat_cols"]

    print("\n" + "=" * 60)
    print(f" ĐANG ĐÁNH GIÁ CHIẾN LƯỢC: {name.upper()} ")
    print(f" Số lượng features đầu vào: {len(current_features)}")
    print("=" * 60)

    oof_preds = np.zeros(len(X_train))
    feature_importance_list = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train[current_features], y_train)
    ):
        print(f"Fold {fold+1}/5...")

        X_tr = X_train[current_features].iloc[train_idx]
        X_val = X_train[current_features].iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model_filtered = lgb.LGBMClassifier(**params_default)
        model_filtered.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
            categorical_feature=current_cat_cols,
        )

        oof_preds[val_idx] = model_filtered.predict_proba(X_val)[:, 1]

        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        fold_gini = fold_auc * 2 - 1
        print(
            f"  Fold {fold+1} — AUC: {fold_auc:.4f} | GINI: {fold_gini:.4f} | Best iteration: {model_filtered.best_iteration_}"
        )

        feature_importance_list.append(
            pd.DataFrame(
                {
                    "feature": current_features,
                    "importance": model_filtered.feature_importances_,
                }
            )
        )

    # Tính toán AUC và GINI trên toàn bộ Out-Of-Fold dữ liệu
    oof_auc = roc_auc_score(y_train, oof_preds)
    oof_gini = oof_auc * 2 - 1

    # Lưu lại kết quả để so sánh cuối cùng
    evaluation_results[name] = {
        "auc": oof_auc,
        "gini": oof_gini,
        "num_features": len(current_features),
    }

# ==============================================================================
# BẢNG TỔNG HỢP SO SÁNH BAO GỒM CẢ 2 PHƯƠNG PHÁP LỌC
# ==============================================================================
print(f"\n{'='*65}")
print(f"{'BẢNG ĐỐI CHIẾU HIỆU NĂNG CÁC MỐC':^65}")
print(f"{'='*65}")
print(f"Mốc 1 — Baseline gốc (265 features, default params):")
print(f"  OOF AUC:  0.7846 | OOF GINI: 0.5693")
print("-" * 65)

for name, res in evaluation_results.items():
    auc_diff = res["auc"] - 0.7846
    feat_reduction = (1 - res["num_features"] / 265) * 100
    print(f"Mốc 2 ({name}) — Filtered ({res['num_features']} features, default params):")
    print(f"  OOF AUC:  {res['auc']:.4f} | OOF GINI: {res['gini']:.4f}")
    print(f"  Chênh lệch AUC vs Mốc 1: {auc_diff:+.4f}")
    print(f"  Số features giảm: 265 → {res['num_features']} ({feat_reduction:.1f}% reduction)")
    print("-" * 65)
print(f"{'='*65}")